Note we utilize duckdb view chains to modify the large dataset

If you are unfamiliar with duckdb please refer to the documentation: https://duckdb.org/docs/stable/clients/python/overview

# Data Loading

## Set root folder path 

In [18]:
import os
from pathlib import Path

target_root = Path(r"C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao")

# Check if it exists before moving
if target_root.exists():
    os.chdir(target_root)
    print(f"✅ Success! Moved to: {Path.cwd()}")
else:
    print(f"❌ Error: The folder '{target_root}' does not exist.")

✅ Success! Moved to: C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao


In [19]:
project_root = target_root

root_folder =  project_root / "downloads/data/raw/"
filtered_output_root_folder = project_root / "downloads/data/filtered/"

services_path = root_folder / "NS-services/services_merged_all_ns_only.parquet"
disruptions_path = root_folder / "NS-disruptions/disruptions_merged_all.parquet"
stations_path = root_folder / "NS-stations/stations-2023-09-nl.csv" #duckdb seem to handle 'NA' station code properly, so load the original file
station_distances_path = root_folder / "NS-tariff-distances/tariff-distances-2022-01.csv" #probably not used in eda, might only be useful for graph edges feature
stations_connections_path = root_folder / "railway_map/connection_edges.parquet" 

weather_path = root_folder / "weather/weather_merged_all.parquet"
holiday_path = root_folder / "holidays/dutch_holidays_2019_2025.parquet"

# 1. Setup paths to iterate over

paths = {
    "Services": services_path,
    "Disruptions": disruptions_path,
    "Stations": stations_path,
    "Station Distances": station_distances_path,
    "Weather": weather_path,
    "Holidays": holiday_path
}


## Duckdb prep

In [20]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path

RUN CELL BELOW ONCE

In [21]:
# 2. Initialize Disk-Based Database
# This creates a file 'main_thesis_data.duckdb' in your current folder.
# Intermediate calculations (joins/group bys) will spill here instead of crashing RAM.
con = duckdb.connect(database='main_thesis_data.duckdb') 

# 3. Register Views (Zero-Copy), Run once
# We use VIEWs so we don't duplicate the Parquet data into the .duckdb file.
# DuckDB reads directly from the parquet files on demand.
con.execute(f"CREATE OR REPLACE VIEW raw_services AS SELECT * FROM read_parquet('{services_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_disruptions AS SELECT * FROM read_parquet('{disruptions_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_weather AS SELECT * FROM read_parquet('{weather_path}')")
con.execute(f"CREATE OR REPLACE VIEW stations AS SELECT * FROM read_csv_auto('{stations_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_holidays AS SELECT * FROM read_parquet('{holiday_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_station_distances AS SELECT * FROM read_csv_auto('{station_distances_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_stations_connections AS SELECT * FROM read_parquet('{stations_connections_path}')") # Prob not needed as stations from RdT is pulled from NS API already, but just in case

# 4. Create CLEAN Views (Logic remains the same, but executed safely)
con.execute("""
    CREATE OR REPLACE VIEW services AS 
    SELECT 
        "Service:RDT-ID" AS service_id,
        "Service:Date" AS service_date,
        "Service:Type" AS train_type,
            
        "Service:Completely cancelled" AS is_completely_cancelled, 
        "Service:Partly cancelled" AS is_partly_cancelled,

        "Stop:Station code" AS station_code,
        "Stop:Station name" AS station_name,
        
        -- Apply Timezone conversion HERE in the SELECT statement
        ("Stop:Arrival time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS arrival_time,
        
        "Stop:Arrival delay" AS arrival_delay_min,
        "Stop:Arrival cancelled" AS is_arrival_cancelled,
        
        -- Apply Timezone conversion HERE too
        ("Stop:Departure time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS departure_time,
        
        "Stop:Departure delay" AS departure_delay_min,
        "Stop:Departure cancelled" AS is_departure_cancelled,
        
        "Stop:Platform change" AS has_platform_change,
            
    FROM raw_services
""")

print("✅ Table 'services' created with clean names and local timestamps.")

con.execute("""
    CREATE OR REPLACE VIEW disruptions AS 
    SELECT 
        *,
        CASE 
            WHEN rdt_station_codes LIKE '[%]' 
            THEN string_split(trim(rdt_station_codes, '[]'), ', ') 
            ELSE string_split(rdt_station_codes, ',') 
        END AS station_array
    FROM raw_disruptions
""")

print("Disk-based database initialized. Queries will now spill to 'main_thesis_data.duckdb' if needed.")

✅ Table 'services' created with clean names and local timestamps.
Disk-based database initialized. Queries will now spill to 'main_thesis_data.duckdb' if needed.


In [22]:
db_name = 'main_thesis_data.duckdb'

# Check before
print(f"Does file exist? {os.path.exists(db_name)}")

con = duckdb.connect(database='main_thesis_data.duckdb') 

Does file exist? True


## Total services in dataset

Total services rowcount = 83816585

In [23]:
display(con.execute("SELECT count(service_id) FROM services").df())

,count(service_id)
0,83816585


## Checking na values

### NA values in services meaning

In [28]:
# Check if there are cases where arrival_delay_min is present but arrival_time is NULL (which would mean that a train was delayed but not scheduled to arrive)
display(con.execute("""
    SELECT 
        service_id,
        station_name,
        arrival_time,
        arrival_delay_min,
        is_arrival_cancelled
    FROM services
    WHERE 
        arrival_delay_min IS NOT NULL 
        AND arrival_time IS NULL
    LIMIT 20
""").df())

,service_id,station_name,arrival_time,arrival_delay_min,is_arrival_cancelled


In [25]:
# Query to find "Terminus" stops (Arrival exists, Departure missing)
df_terminus = con.execute("""
    SELECT 
        service_id,
        station_name,
        arrival_delay_min,
        arrival_time,
        is_arrival_cancelled
    FROM services
    WHERE 
        -- Condition 1: Departure info is missing (OR logic as requested)
        (arrival_time IS NULL OR arrival_delay_min IS NULL OR is_arrival_cancelled IS NULL)
        
        AND 
        
        -- Condition 2: Arrival info is present (OR logic as requested)
        (departure_time IS NOT NULL OR departure_delay_min IS NOT NULL OR is_departure_cancelled IS NOT NULL)
    
    -- Optional: Limit to see a sample first
    LIMIT 20
""").df()

display(df_terminus)

,service_id,station_name,arrival_delay_min,arrival_time,is_arrival_cancelled
0,744694,Amsterdam Centraal,<NA>,NaT,<NA>
1,744697,Rhenen,<NA>,NaT,<NA>
2,744699,Baarn,<NA>,NaT,<NA>
3,744700,Dordrecht,<NA>,NaT,<NA>
4,744701,Leiden Centraal,<NA>,NaT,<NA>
5,744702,Leiden Centraal,<NA>,NaT,<NA>
6,744703,Vlissingen,<NA>,NaT,<NA>
7,744704,'s-Hertogenbosch,<NA>,NaT,<NA>
8,744705,Utrecht Centraal,<NA>,NaT,<NA>
9,744706,Woerden,<NA>,NaT,<NA>


#### p.1) Handling NA values
- Remove unscheduled trains (NA at both arrival and departure columns)
- Fill start stations (NA at arrival columns, it never 'arrives' so columns:arrival_time = departure_time, arrival_delay_min = 0, is_arrival_cancelled = FALSE)
- Fill end stations (NA at departure columns, it never 'departs' so columns: departure_time = arrival_time, departure_delay_min = 0, is_departure_cancelled = FALSE)

##### A) Removing unscheduled services (services with ONLY NA values) (5652 services)

In [26]:
con.execute("""
    CREATE OR REPLACE VIEW services_scheduled AS 
    SELECT * FROM services
    WHERE NOT (
        arrival_time IS NULL 
        AND departure_time IS NULL 
        AND arrival_delay_min IS NULL
        AND departure_delay_min IS NULL
    )
""")

print("✅ View 'services_scheduled' created (Unscheduled rows hidden).")

✅ View 'services_scheduled' created (Unscheduled rows hidden).


scheduled services rowcount = 83810933

In [54]:
display(con.execute("SELECT count(service_id) FROM services_scheduled").df())

,count(service_id)
0,83810933


##### B) Imputing NA values for arrival columns (8_121_562 services) and departure columns (8_119_198 services)

2364 train services are missing end stations, most likely could be explained by the column "Service:Train number" due to merging (or splitting) of trains.
"A single service may sometimes have multiple train numbers. For example, when a train is split in two parts, or when a train changes a train number on a major station halfway." https://www.rijdendetreinen.nl/en/open-data/train-archive

In [47]:
print(8_121_562 - 8_119_198)

2364


In [50]:
con.execute("""
    CREATE OR REPLACE VIEW services_filled AS 
    SELECT 
        * REPLACE (
            -- =========================================================
            -- 1. STRICT Fix for Start Stations (Fill Arrival)
            -- Only runs if ALL 3 arrival columns are NULL
            -- =========================================================
            CASE 
                WHEN arrival_time IS NULL 
                     AND arrival_delay_min IS NULL 
                     AND is_arrival_cancelled IS NULL
                THEN departure_time             -- Action: Fill with Departure
                ELSE arrival_time               -- Action: Keep original (even if NULL)
            END AS arrival_time,

            CASE 
                WHEN arrival_time IS NULL 
                     AND arrival_delay_min IS NULL 
                     AND is_arrival_cancelled IS NULL
                THEN 0                          -- Action: Assume 0 delay
                ELSE arrival_delay_min          -- Action: Keep original
            END AS arrival_delay_min,

            CASE 
                WHEN arrival_time IS NULL 
                     AND arrival_delay_min IS NULL 
                     AND is_arrival_cancelled IS NULL
                THEN FALSE                      -- Action: Assume not cancelled
                ELSE is_arrival_cancelled       -- Action: Keep original
            END AS is_arrival_cancelled,

            -- =========================================================
            -- 2. STRICT Fix for End Stations (Fill Departure)
            -- Only runs if ALL 3 departure columns are NULL
            -- =========================================================
            CASE 
                WHEN departure_time IS NULL 
                     AND departure_delay_min IS NULL 
                     AND is_departure_cancelled IS NULL
                THEN arrival_time               -- Action: Fill with Arrival
                ELSE departure_time             -- Action: Keep original
            END AS departure_time,

            CASE 
                WHEN departure_time IS NULL 
                     AND departure_delay_min IS NULL 
                     AND is_departure_cancelled IS NULL
                THEN 0 
                ELSE departure_delay_min 
            END AS departure_delay_min,

            CASE 
                WHEN departure_time IS NULL 
                     AND departure_delay_min IS NULL 
                     AND is_departure_cancelled IS NULL
                THEN FALSE 
                ELSE is_departure_cancelled 
            END AS is_departure_cancelled
        )
    FROM services_scheduled
""")

print("✅ View 'services_filled' created. Strict imputation applied.")

✅ View 'services_filled' created. Strict imputation applied.


### Arrival Delay of Cancellations

We set an arrival delay penalty of 30 min to partly cancelled services and 60 min to completely cancelled services

Cancellations as % of all services

In [62]:
con.execute("""
    SELECT 
        -- Total
        count(*) as total_services,
        
        -- Counts
        count(*) FILTER (WHERE is_partly_cancelled OR is_completely_cancelled) as total_all_cancelled_services,
        count(*) FILTER (WHERE is_partly_cancelled AND NOT is_completely_cancelled) as count_partly_cancelled,
        count(*) FILTER (WHERE is_completely_cancelled) as count_completely_cancelled,
        count(*) FILTER (WHERE is_partly_cancelled AND is_completely_cancelled) as count_both_cancelled,
            
            
        -- Percentages (Logic repeated)
        round(
            total_all_cancelled_services / total_services * 100.0, 
        2) as pct_all_cancelled,

        round(
            count_partly_cancelled / total_services * 100.0, 
        2) as pct_partly_cancelled,

        round(
            count_completely_cancelled / total_services * 100.0, 
        2) as pct_completely_cancelled,
            
        round(
            count_both_cancelled / total_services * 100.0,
        2) as pct_both_cancelled
        
    FROM services_filled
""").df()

,total_services,total_all_cancelled_services,count_partly_cancelled,count_completely_cancelled,count_both_cancelled,pct_all_cancelled,pct_partly_cancelled,pct_completely_cancelled,pct_both_cancelled
0,83810933,7813345,5945583,1867762,1859145,9.32,7.09,2.23,2.22


Cancellations as % of all CANCELLED services

In [63]:
con.execute("""
    SELECT 
        -- Total
        -- count(*) as total_services,
        
        -- Counts
        count(*) FILTER (WHERE is_partly_cancelled OR is_completely_cancelled) as total_all_cancelled_services,
        count(*) FILTER (WHERE is_partly_cancelled AND NOT is_completely_cancelled) as count_partly_cancelled,
        count(*) FILTER (WHERE is_completely_cancelled) as count_completely_cancelled,
        count(*) FILTER (WHERE is_partly_cancelled AND is_completely_cancelled) as count_both_cancelled,
            
        -- Percentages (Logic repeated)
        round(
            total_all_cancelled_services / total_all_cancelled_services * 100.0, 
        2) as pct_all_cancelled,

        round(
            count_partly_cancelled / total_all_cancelled_services * 100.0, 
        2) as pct_partly_cancelled,

        round(
            count_completely_cancelled / total_all_cancelled_services * 100.0, 
        2) as pct_completely_cancelled,
        
        round(
            count_both_cancelled / total_all_cancelled_services * 100.0,
        2) as pct_both_cancelled
    
        
    FROM services_filled
""").df()

,total_all_cancelled_services,count_partly_cancelled,count_completely_cancelled,count_both_cancelled,pct_all_cancelled,pct_partly_cancelled,pct_completely_cancelled,pct_both_cancelled
0,7813345,5945583,1867762,1859145,100.0,76.1,23.9,23.79


9.3% (7_813_345 million services) of all services (83_810_933) are affected by cancellations

out of all cancelled services

76.1% is partly cancelled

23.9% is completely cancelled

In [64]:
print(7_813_345 / 83_810_933 * 100)

9.322584441340128


In [ ]:
con.execute("""
    SELECT 
        -- Total Cancelled Rows
        count(*) as total_cancelled,
        
        -- 1. Delay >= 60 mins
        count(*) FILTER (WHERE arrival_delay_min >= 60) as count_gt_60,
        round(count(*) FILTER (WHERE arrival_delay_min >= 60) * 100.0 / count(*), 2) as pct_gt_60,

        -- 2. Delay >= 30 < 60 mins
        count(*) FILTER (WHERE arrival_delay_min >= 30 AND arrival_delay_min < 60) as count_gt_30,
        round(count(*) FILTER (WHERE arrival_delay_min >= 30 AND arrival_delay_min < 60) * 100.0 / count(*), 2) as pct_gt_30,
        
        -- 2. Delay > 0 < 30 mins
        count(*) FILTER (WHERE arrival_delay_min > 0 AND arrival_delay_min < 30) as count_gt_1,
        round(count(*) FILTER (WHERE arrival_delay_min > 0 AND arrival_delay_min < 30) * 100.0 / count(*), 2) as pct_gt_1,

        -- 3. Delay <= 0 (Likely "Clean" cancellations)
        count(*) FILTER (WHERE arrival_delay_min <= 0) as count_eq_0,
        round(count(*) FILTER (WHERE arrival_delay_min <= 0) * 100.0 / count(*), 2) as pct_eq_0,
            
    FROM services_filled
    WHERE 
        -- Focus only on cancelled services
        (is_partly_cancelled = TRUE OR is_completely_cancelled = TRUE)
""").df()

,total_cancelled,count_gt_60,pct_gt_60,count_gt_30,pct_gt_30,count_gt_1,pct_gt_1,count_eq_0,pct_eq_0,sum_pct
0,7813345,3197,0.04,20864,0.27,1124089,14.39,6665195,85.31,100.01


#### p.2) Set Arrival Delay Penalty for both completely and partly cancelled services (changes the delayed classification distribution)

In [73]:
con.execute("""
    CREATE OR REPLACE VIEW services_penalty AS 
    SELECT * REPLACE (
        CASE
            WHEN is_completely_cancelled = TRUE THEN 60
            WHEN is_partly_cancelled = TRUE THEN 30
            ELSE arrival_delay_min
        END AS arrival_delay_min
    )
    FROM services_filled
""")

# Verify the "overwrite" worked
con.execute("SELECT is_completely_cancelled, arrival_delay_min FROM services_penalty WHERE is_completely_cancelled = TRUE LIMIT 5").df()

,is_completely_cancelled,arrival_delay_min
0,True,60
1,True,60
2,True,60
3,True,60
4,True,60


In [74]:
display(con.execute("SELECT * FROM services_penalty LIMIT 10").df())

,service_id,service_date,train_type,is_completely_cancelled,is_partly_cancelled,station_code,station_name,arrival_time,arrival_delay_min,is_arrival_cancelled,departure_time,departure_delay_min,is_departure_cancelled,has_platform_change
0,744693,2019-01-02,Sprinter,False,False,APD,Apeldoorn,2019-01-02 09:40:00,0,False,2019-01-02 09:40:00,0,False,False
1,744694,2019-01-02,Intercity direct,False,False,ASD,Amsterdam Centraal,2019-01-02 08:22:00,0,False,2019-01-02 08:22:00,0,False,False
2,744694,2019-01-02,Intercity direct,False,False,SHL,Schiphol Airport,2019-01-02 08:35:00,0,False,2019-01-02 08:37:00,2,False,True
3,744694,2019-01-02,Intercity direct,False,False,RTD,Rotterdam Centraal,2019-01-02 09:03:00,1,False,2019-01-02 09:03:00,0,False,False
4,744697,2019-01-02,Sprinter,False,False,RHN,Rhenen,2019-01-02 08:22:00,0,False,2019-01-02 08:22:00,0,False,False
5,744697,2019-01-02,Sprinter,False,False,VNDC,Veenendaal Centrum,2019-01-02 08:28:00,0,False,2019-01-02 08:30:00,0,False,False
6,744697,2019-01-02,Sprinter,False,False,VNDW,Veenendaal West,2019-01-02 08:32:00,0,False,2019-01-02 08:32:00,1,False,False
7,744697,2019-01-02,Sprinter,False,False,MRN,Maarn,2019-01-02 08:42:00,0,False,2019-01-02 08:42:00,1,False,False
8,744697,2019-01-02,Sprinter,False,False,DB,Driebergen-Zeist,2019-01-02 08:48:00,0,False,2019-01-02 08:48:00,1,False,False
9,744697,2019-01-02,Sprinter,False,False,BNK,Bunnik,2019-01-02 08:52:00,0,False,2019-01-02 08:52:00,1,False,False


could also exclude all services with cancellations from the services dataset with the commented code below

In [ ]:
# con.execute("""
#     CREATE OR REPLACE VIEW services_penalty AS 
#     SELECT * FROM services_filled
#     WHERE 
#         -- Exclude cancelled arrivals
#         is_arrival_cancelled IS NOT TRUE
        
#         -- Exclude cancelled departures
#         AND is_departure_cancelled IS NOT TRUE
# """)

# # Quick verification: Check how many rows remain
# print(con.execute("SELECT count(*) FROM services_penalty").fetchone()[0])

## Transformation 1: adding 'to stations' columns to services dataset

Each service has atleast two rows: the departure (current Stop:Station, i.e. from station) and arrival (to station inferred from sorting the service line). 

NA values in arrival_time and arrival_delay_min indicate the start of a service.
NA values in departure_time and arrival_delay_min indicate the end of a service. 

A starting station could never have arrival delay as it is already there (IS NA values), so we set this to 0

In [75]:
# ==========================================
# STEP 2: Create Edges (Window Functions)
# ==========================================
# We materialize this as a TABLE to compute the expensive window functions once.
con.execute("""
    CREATE OR REPLACE VIEW services_with_edges AS 
    SELECT 
        *,
        -- From station is just the current row's station code
        station_code AS source,
            
        -- 1. Where are we going next? Get Destination (Next Row)
        LEAD(station_code) OVER (
            PARTITION BY service_id 
            ORDER BY COALESCE(departure_time, arrival_time) ASC
        ) AS target,

        -- 2. When do we get there? Get Arrival Time (Next Row)
        LEAD(arrival_time) OVER (
            PARTITION BY service_id 
            ORDER BY COALESCE(departure_time, arrival_time) ASC
        ) AS scheduled_arrival_time

    FROM services_penalty
""")
print("✅ Layer 2: Added Edge columns.")

✅ Layer 2: Added Edge columns.


Edges = 2931 (takes ~1min locally)

In [21]:
# display(con.execute("""
#     SELECT count(*) AS unique_edges
#     FROM (
#         SELECT DISTINCT source, target 
#         FROM services_with_edges
#         WHERE target IS NOT NULL
#     )
# """).df())

## Transformation 2: Stations
We keep the stations dataset as ground truth for which stations exist

### A) Add station distances feature

In [77]:
# ==========================================
# STEP 1: Flatten distance matrix dataset
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW station_distances_long AS 
    SELECT 
        Station AS source,          -- Rename 'Station' to 'source' for clarity
        target_station AS target, 
        CAST(distance AS INTEGER) AS distance
    FROM (
        UNPIVOT raw_station_distances
        ON COLUMNS(* EXCLUDE (Station))
        INTO
            NAME target_station
            VALUE distance
    )
    -- Filter 1: Explicitly remove the known garbage values
    WHERE distance NOT IN ('?', 'XXX')
    
    -- Filter 2: (Optional but Safer) Ensure only valid numbers remain
    -- AND TRY_CAST(distance AS INTEGER) IS NOT NULL
""")

# Let's verify the columns first to be safe:
print("Unpivoted columns:", con.execute("DESCRIBE station_distances_long").df()['column_name'].tolist())

Unpivoted columns: ['source', 'target', 'distance']


In [ ]:
con.execute("""
    -- 1. Get ALL stations from the matrix
    WITH matrix_stations AS (
        SELECT TRIM(source) AS station FROM station_distances_long
        UNION 
        SELECT TRIM(target) AS station FROM station_distances_long
    )
    
    -- 2. "Subtract" the official list
    SELECT station AS not_in_stations FROM matrix_stations
    
    EXCEPT 
    
    SELECT TRIM(code) FROM stations
""").df()

,not_in_stations
0,WR
1,LEER


In [ ]:
con.execute("""
    SELECT TRIM(code) AS not_in_distances
    FROM stations
    
    EXCEPT 
    
    -- Subtract all stations found in the matrix
    (
        SELECT TRIM(source) FROM station_distances_long
        UNION 
        SELECT TRIM(target) FROM station_distances_long
    )
""").df()

,not_in_distances


In [72]:
# ==========================================
# STEP 2: Join Distances
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW services_edges_distances AS 
    SELECT 
        e.*, 
        d.distance AS distance
    FROM services_with_edges e
    LEFT JOIN station_distances_long d
        ON e.source = d.source 
        AND e.target = d.target
""")

print("✅ Layer 2A: Distances joined.")

✅ Layer 2A: Distances joined.


330 edges missing distances i.e. not in stations distances dataset ~(1 min)

In [ ]:
# # Check for "Orphan Edges" (Edges that exist but have no distance)
# orphan_edges = con.execute("""
#     SELECT DISTINCT source, target 
#     FROM services_edges_distances
#     WHERE 
#         target IS NOT NULL      -- Ignore the last station of a trip (it has no target)
#         AND distance IS NULL    -- Capture where the join failed
# """).df()

# if not orphan_edges.empty:
#     print(f"⚠️ Warning: {len(orphan_edges)} routes are missing distance data.")
#     display(orphan_edges.head())
# else:
#     print("✅ Success: All edges have a corresponding distance.")

⚠️ Warning: 330 routes are missing distance data.


,source,target
0,BD,NDKP
1,ATW,BERCH
2,FBNL,BRUSN
3,BRUSZ,BRUSC
4,MINDEN,OEYNH


### B) Services with valid stations

In [30]:
# ==========================================
# Filter Invalid Rows
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW services_valid AS 
    SELECT * FROM services_edges_distances
    WHERE 
        -- 1. Distance must be known (removes missing edges from the Matrix)
        distance IS NOT NULL
        
        -- 2. Source must be in the official station list
        AND (source IN (SELECT code FROM stations)
        
        -- 3. Target must be in the official station list
        AND target IN (SELECT code FROM stations))
""")

print("✅ Layer 2B: Filtered services to valid edges only.")

✅ Layer 2B: Filtered services to valid edges only.


Valid services rowcount = 
75308674

In [ ]:
# display(con.execute("SELECT count(service_id) FROM services_valid").df())

,count(service_id)
0,75308674


### Aggregate and add hourly featrures at the end 
Holidays, weather, delay (classification) column 

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services AS 
    WITH base_transform AS (
        SELECT 
            "Service:RDT-ID" AS service_id,
            "Service:Date" AS service_date,
            "Service:Type" AS train_type,
            "Stop:Station code" AS station_code,
            "Stop:Station name" AS station_name,
            
            -- Timezone conversions
            ("Stop:Arrival time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS arrival_time,
            ("Stop:Departure time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS departure_time,
            
            -- Ensure delay is treated as a number for the logic
            TRY_CAST("Stop:Arrival delay" AS DOUBLE) AS arrival_delay_min,
            "Stop:Arrival cancelled" AS is_cancelled
        FROM raw_services
    )
    SELECT 
        *,
        -- CLASSIFICATION FEATURE FLAG: is_delayed
        CASE 
            WHEN arrival_delay_min IS NULL THEN NULL
            WHEN arrival_delay_min > 0 THEN 1
            ELSE 0
        END AS is_delayed,
        
    FROM base_transform
""")

print("✅ 'services' view updated with 'is_delayed' flag (preserving NAs).")

✅ 'services' view updated with 'is_delayed' flag (preserving NAs).


# Feature Engineering

In [ ]:
# ==========================================
# STEP 2: Create Edges (Window Functions)
# ==========================================
# We materialize this as a TABLE to compute the expensive window functions once.
print("Step 2: Linking stops to create edges...")
con.execute("""
    CREATE OR REPLACE TABLE edges_raw AS
    SELECT 
        "Service:RDT-ID",
        "Service:Type",
        "Stop:Station code" AS from_station,
        
        -- Get Destination (Next Row)
        LEAD("Stop:Station code") OVER (
            PARTITION BY "Service:RDT-ID" 
            ORDER BY "Stop:Arrival time" ASC
        ) AS to_station,
        
        -- Get Arrival Time (Next Row)
        LEAD("Stop:Arrival time") OVER (
            PARTITION BY "Service:RDT-ID" 
            ORDER BY "Stop:Arrival time" ASC
        ) AS arrival_time,
        
        -- Get Delay (Next Row)
        LEAD("Stop:Arrival delay") OVER (
            PARTITION BY "Service:RDT-ID" 
            ORDER BY "Stop:Arrival time" ASC
        ) AS arrival_delay,

        -- Get Cancellation Status (Next Row)
        LEAD("Stop:Arrival cancelled") OVER (
            PARTITION BY "Service:RDT-ID" 
            ORDER BY "Stop:Arrival time" ASC
        ) AS is_cancelled

    FROM raw_services
    WHERE "Stop:Arrival time" IS NOT NULL;
""")

Step 2: Linking stops to create edges...


In [ ]:
# Check data
print(con.execute("SELECT * FROM edges_raw LIMIT 10").df())

Setting cancellations as 60 minutes arrival delay (introduces bias)

Q -> create hourly bins or 15 mins to not lose

In [ ]:
bin_timehorizon = "1 hour"
# bin_timehorizon = "15 minutes"

In [ ]:
# ==========================================
# STEP 3: Clean & Bucket Time
# ==========================================
print("Step 3: Cleaning edges and creating 15-min buckets...")
con.execute(f"""
    CREATE OR REPLACE TABLE edges_cleaned AS
    SELECT 
        -- Unique ID: ASD_UT_Intercity
        from_station || '_' || to_station || '_' || "Service:Type" AS unique_id,
        
        -- Time Bucket: Floor to nearest bin interval
        time_bucket(INTERVAL '{bin_timehorizon}', CAST(arrival_time AS TIMESTAMP)) AS time_idx,
        
        -- Impute Delay: 60 mins if cancelled, else actual delay
        CASE 
            WHEN is_cancelled = true THEN 60.0 
            ELSE COALESCE(arrival_delay, 0.0) 
        END AS delay_value,
        
        -- Extract Service Type for static feature later
        "Service:Type" as service_type
    FROM edges_raw
    WHERE to_station IS NOT NULL;
""")

Step 3: Cleaning edges and creating 15-min buckets...


In [ ]:
# ==========================================
# STEP 4: Build the "Skeleton" (Master Grid)
# ==========================================
print("Step 4: Generating the master time-series skeleton...")

# 4a. Get all unique IDs
con.execute("CREATE OR REPLACE TABLE distinct_ids AS SELECT DISTINCT unique_id, service_type FROM edges_cleaned")

# 4b. Get the full time range (Min to Max)
con.execute(f"""
    CREATE OR REPLACE TABLE all_times AS 
    SELECT generate_series(
        (SELECT MIN(time_idx) FROM edges_cleaned), 
        (SELECT MAX(time_idx) FROM edges_cleaned), 
        INTERVAL '{bin_timehorizon}'
    ) AS time_idx
""")

# 4c. Cartesian Product (Cross Join)
con.execute("""
    CREATE OR REPLACE TABLE skeleton AS
    SELECT 
        t.time_idx, 
        i.unique_id,
        i.service_type -- Keep static features in the skeleton
    FROM all_times t 
    CROSS JOIN distinct_ids i
""")

Step 4: Generating the master time-series skeleton...


RuntimeError: Query interrupted

In [ ]:
# ==========================================
# STEP 5: Aggregate Observations
# ==========================================
print("Step 5: Aggregating observations per bucket...")
con.execute("""
    CREATE OR REPLACE TABLE observations_agg AS
    SELECT 
        unique_id,
        time_idx,
        MAX(delay_value) as max_delay,
        COUNT(*) as traffic_volume
    FROM edges_cleaned
    GROUP BY 1, 2
""")

In [ ]:
# ==========================================
# STEP 6: Final Merge & Export
# ==========================================
print("Step 6: Merging and exporting...")
con.execute("""
    CREATE OR REPLACE TABLE final_dataset AS
    SELECT 
        s.time_idx,
        s.unique_id,
        
        -- Target: Fill gaps with 0.0
        COALESCE(o.max_delay, 0.0) AS target_delay,
        
        -- Dynamic Covariate: Fill gaps with 0 volume
        COALESCE(o.traffic_volume, 0) AS traffic_volume,
        
        -- Static Covariate
        s.service_type
        
    FROM skeleton s
    LEFT JOIN observations_agg o 
        ON s.time_idx = o.time_idx 
        AND s.unique_id = o.unique_id
    ORDER BY s.unique_id, s.time_idx
""")

In [ ]:
# Check final data
print(con.execute("SELECT * FROM final_dataset LIMIT 10").df())

In [ ]:
# Save to Parquet
con.execute("COPY final_dataset TO 'train_time_series.parquet' (FORMAT PARQUET)")
print("Done! Saved to train_time_series.parquet")

# [OLD] Creating full time series dataset

### Creating main time range foundation

First we create the start and end hourly time range (we also add numerical representations for sin/cos transformations later)

NOTE time range is currenlty based on services where we utilize arrival_time as bounds, but could use service_date instead if needed (which essentially adds some empty hours at the start and end, but is prob more straightforward time range)

In [ ]:
# --- Cell: Step 1 - Create Time Spine with Calendar Features ---

print("Creating continuous hourly timeline with calendar features...")

# We use a Common Table Expression (CTE) 'series' to generate the list first,
# then we select from it to calculate the extra columns easily.

create_main_query = """
    CREATE OR REPLACE TABLE timetable_main AS
    WITH bounds AS (
        SELECT 
            date_trunc('hour', MIN(arrival_time)) as start_time,
            date_trunc('hour', MAX(arrival_time)) as end_time
        FROM services
    ),
    raw_series AS (
        SELECT 
            UNNEST(GENERATE_SERIES(start_time, end_time, INTERVAL 1 HOUR)) as hour_bin
        FROM bounds
    )
    SELECT 
        hour_bin,
        -- Extract Time Features (Numerical)
        HOUR(hour_bin) as hour_of_day,      -- 0 to 23
        ISODOW(hour_bin) as day_of_week,    -- 1 (Monday) to 7 (Sunday)
        MONTH(hour_bin) as month_of_year    -- 1 to 12
    FROM raw_series
    ORDER BY hour_bin
"""

con.execute(create_main_query)


# Converting time zone info
convert_tz_query = """
    ALTER TABLE timetable_main 
    ALTER COLUMN hour_bin TYPE TIMESTAMP
"""

con.execute(convert_tz_query)

# --- Verification ---
print("✅ Table 'timetable_main' created.")
print(con.execute("SELECT * FROM timetable_main LIMIT 5").df())

# Check total range
check_query = """
    SELECT 
        MIN(hour_bin) as start, 
        MAX(hour_bin) as end, 
        COUNT(*) as total_hours
    FROM timetable_main
"""
print("\nRange Statistics:")
print(con.execute(check_query).df())

Creating continuous hourly timeline with calendar features...
✅ Table 'timetable_main' created.
             hour_bin  hour_of_day  day_of_week  month_of_year
0 2019-01-01 02:00:00            2            2              1
1 2019-01-01 03:00:00            3            2              1
2 2019-01-01 04:00:00            4            2              1
3 2019-01-01 05:00:00            5            2              1
4 2019-01-01 06:00:00            6            2              1

Range Statistics:
                start                 end  total_hours
0 2019-01-01 02:00:00 2025-01-01 02:00:00        52609


In [ ]:
# This shows Column Name, Column Type, Nullable, etc.
con.sql(f"DESCRIBE SELECT * FROM timetable_main").show()
# This gives you stats per column, including missing values (null_percentage)
con.sql(f"SUMMARIZE SELECT * FROM timetable_main").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ hour_bin      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ hour_of_day   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ day_of_week   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ month_of_year │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌───────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬─────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬───────┬─────────────────┐
│  column_name  │ column_type │         min         │         max         │ approx_unique │         avg         │        std

### Adding Station Cross Join

Note/todo we currently use all stations present in the stations dataset, cuz currently when filtering on only stations present in the services (non ghost stations) it seems that we disconnect the graph somehow

In [ ]:
# --- Cell: Step 2 - Create Spatio-Temporal Grid ---

print("Step 2: Expanding timeline to all stations (Cross Join)...")

# We take the timetable (with calendar features) and multiply it by every active station.
create_grid_query = """
    CREATE OR REPLACE TABLE timetable_grid AS
    SELECT 
        t.hour_bin,
        -- Carry over the calendar features we created in Step 1
        t.hour_of_day,
        t.day_of_week,
        t.month_of_year,
        
        -- The Station Column
        s.code as station_code,
        s.type as station_type
        
    FROM timetable_main t
    CROSS JOIN stations s
    ORDER BY t.hour_bin, s.code
"""

con.execute(create_grid_query)

# --- Verification ---
# The total rows should be: (Total Hours) * (Total Stations)
stats_query = """
    SELECT 
        (SELECT COUNT(*) FROM timetable_main) as total_hours,
        (SELECT COUNT(*) FROM stations) as total_stations,
        COUNT(*) as grid_rows
    FROM timetable_grid
"""
stats = con.execute(stats_query).df()

print("✅ Table 'timetable_grid' created.")
display(stats)

# Quick sanity check: Did the math work?
expected = stats['total_hours'][0] * stats['total_stations'][0]
actual = stats['grid_rows'][0]

if expected == actual:
    print(f"Verified: Grid size is correct ({actual:,} rows).")
else:
    print(f"⚠️ Mismatch! Expected {expected:,} but got {actual:,}")

Step 2: Expanding timeline to all stations (Cross Join)...
✅ Table 'timetable_grid' created.


,total_hours,total_stations,grid_rows
0,52609,397,20885773


Verified: Grid size is correct (20,885,773 rows).


### Adding services

In [ ]:
# --- Cell: Step 3 - Populate Grid with Actual Data ---

print("Step 3: Joining actual services and filling gaps...")


con.execute("""
    CREATE OR REPLACE TABLE hourly_stats AS
    
    WITH actuals AS (
        -- 1. Aggregating the raw services first
        -- This compresses millions of train rows into (Hour x Station) summaries
        SELECT 
            date_trunc('hour', arrival_time) as hour_bin,
            station_code,
            
            -- Metrics
            COUNT(*) as total_trains,
            SUM(CASE WHEN arrival_delay_min > 0 THEN 1 ELSE 0 END) as delayed_trains_count,
            ROUND(SUM(arrival_delay_min), 1) as total_delay_minutes,
            ROUND(AVG(arrival_delay_min), 1) as avg_delay_minutes
            
        FROM services
        WHERE arrival_time IS NOT NULL 
          AND is_cancelled = false 
        GROUP BY 1, 2
    )
    
    SELECT 
        -- 2. Select everything from our Grid (The Spine)
        grid.hour_bin,
        grid.station_code,
        grid.hour_of_day,
        grid.day_of_week,
        grid.month_of_year,
        
        -- 3. Bring in the Actuals, filling gaps with 0
        COALESCE(act.total_trains, 0) as total_trains,
        COALESCE(act.delayed_trains_count, 0) as delayed_trains_count,
        COALESCE(act.total_delay_minutes, 0) as total_delay_minutes,
        
        -- Special handling for Average: 
        -- If 0 trains ran, average delay is technically undefined, but 0 is safe for ML.
        COALESCE(act.avg_delay_minutes, 0) as avg_delay_minutes
        
    FROM timetable_grid grid
    LEFT JOIN actuals act 
        ON grid.hour_bin = act.hour_bin 
        AND grid.station_code = act.station_code
    ORDER BY grid.hour_bin, grid.station_code
""")

# --- Verification ---
print("✅ Table 'hourly_stats' created successfully.")

# Check 1: Verify a busy time (Utrecht Centraal at 8 AM)
print("\n--- Check: Busy Hour (Should have data) ---")
busy_check = """
    SELECT * FROM hourly_stats 
    WHERE station_code = 'UT' AND hour_of_day = 8 
    LIMIT 3
"""
display(con.execute(busy_check).df())

# Check 2: Verify a quiet time (Utrecht Centraal at 4 AM)
# This confirms that our COALESCE logic worked (should see 0s, not NULLs)
print("\n--- Check: Quiet Hour (Should be 0, not NULL) ---")
quiet_check = """
    SELECT * FROM hourly_stats 
    WHERE station_code = 'UT' AND hour_of_day = 4 
    LIMIT 3
"""
display(con.execute(quiet_check).df())

Step 3: Joining actual services and filling gaps...
✅ Table 'hourly_stats' created successfully.

--- Check: Busy Hour (Should have data) ---


,hour_bin,station_code,hour_of_day,day_of_week,month_of_year,total_trains,delayed_trains_count,total_delay_minutes,avg_delay_minutes
0,2019-01-01 08:00:00,UT,8,2,1,34,13.0,60.0,1.8
1,2019-01-02 08:00:00,UT,8,3,1,58,16.0,42.0,0.7
2,2019-01-03 08:00:00,UT,8,4,1,60,10.0,26.0,0.4



--- Check: Quiet Hour (Should be 0, not NULL) ---


,hour_bin,station_code,hour_of_day,day_of_week,month_of_year,total_trains,delayed_trains_count,total_delay_minutes,avg_delay_minutes
0,2019-01-01 04:00:00,UT,4,2,1,1,0.0,0.0,0.0
1,2019-01-02 04:00:00,UT,4,3,1,1,1.0,3.0,3.0
2,2019-01-03 04:00:00,UT,4,4,1,1,0.0,0.0,0.0


### Adding disruptions

In [ ]:
# --- Cell: Step 4 - Add Disruption Features ---

print("Step 4: Merging disruption events into the timeline...")

con.execute("""
    CREATE OR REPLACE TABLE hourly_stats_enriched AS
    
    WITH 
    -- 1. Prepare Disruptions (Ensure we have the flattened view from earlier)
    flat_disruptions AS (
        SELECT 
            UNNEST(station_array) as station_code,
            start_time,
            end_time,
            cause_nl
        FROM disruptions -- Utilizing the view we created in the Quality Check step
        WHERE duration_minutes > 0
    ),
    
    -- 2. Aggregate Disruptions per Hour/Station
    -- (A station might have 2 separate disruptions in one hour)
    hourly_disruptions AS (
        SELECT 
            h.hour_bin,
            h.station_code,
            -- Count how many distinct disruption events touch this hour
            COUNT(d.start_time) as disruption_count,
            -- Boolean flag: Was there ANY disruption?
            MAX(CASE WHEN d.start_time IS NOT NULL THEN 1 ELSE 0 END) as is_disrupted
        FROM hourly_stats h
        LEFT JOIN flat_disruptions d
          ON h.station_code = d.station_code
          -- Overlap Logic: The disruption overlaps with the hour bin
          AND d.start_time < (h.hour_bin + INTERVAL 1 HOUR)
          AND d.end_time > h.hour_bin
        GROUP BY 1, 2
    )
    
    -- 3. Join back to main table
    SELECT 
        base.*,
        COALESCE(d.disruption_count, 0) as disruption_count,
        CAST(COALESCE(d.is_disrupted, 0) as BOOLEAN) as is_disrupted
    FROM hourly_stats base
    LEFT JOIN hourly_disruptions d
        ON base.hour_bin = d.hour_bin 
        AND base.station_code = d.station_code
    ORDER BY base.hour_bin, base.station_code
""")

# --- Verification ---
print("✅ Table 'hourly_stats_enriched' created.")

# Check: Show me an hour with a disruption
print("\n--- Sample: Disrupted Hours ---")
check_query = """
    SELECT hour_bin, station_code, total_trains, avg_delay_minutes, is_disrupted
    FROM hourly_stats_enriched
    WHERE is_disrupted = TRUE
    LIMIT 5
"""
display(con.execute(check_query).df())

Step 4: Merging disruption events into the timeline...
✅ Table 'hourly_stats_enriched' created.

--- Sample: Disrupted Hours ---


,hour_bin,station_code,total_trains,avg_delay_minutes,is_disrupted
0,2019-01-01 06:00:00,GDM,0,0.0,True
1,2019-01-01 07:00:00,ALMO,0,0.0,True
2,2019-01-01 08:00:00,ALMO,0,0.0,True
3,2019-01-01 08:00:00,CL,5,6.6,True
4,2019-01-01 08:00:00,EML,3,1.3,True


### Adding weather data

In [ ]:
# --- Cell: Step 5 - Add Detailed Weather Features ---

print("Step 5: Integrating detailed weather data and calculating lag features...")

con.execute("""
    CREATE OR REPLACE TABLE hourly_stats_final AS
    
    WITH 
    -- 1. Clean & Aggregate Weather to Hourly level as some report by some minor deviations
    weather_hourly AS (
        SELECT
            station_code,
            -- Truncate weather timestamp to the hour to match hour_bin
            DATE_TRUNC('hour', "time") as weather_hour,
            
            -- [Instant Variables] -> Average them for the hour block
            AVG(temperature_2m) as temp_c,
            AVG(wind_speed_10m) as wind_speed_kmh,
            AVG(soil_temperature_0_to_7cm) as soil_temp_c,
            AVG(snow_depth) as snow_depth_m,
            
            -- [Max/Gusts] -> Maximum observed in the hour
            MAX(wind_gusts_10m) as wind_gust_kmh,
            
            -- [Accumulated Variables] -> Sum them (Preceding hour sum)
            COALESCE(SUM(rain), 0) as rain_mm,
            COALESCE(SUM(snowfall), 0) as snow_cm,
            
            -- Boolean Flag: Was there any precipitation this hour? Either raining or snowing
            MAX(CASE WHEN rain > 0 OR snowfall > 0 THEN 1 ELSE 0 END) as is_precipitating_int
        FROM raw_weather
        GROUP BY 1, 2
    ),
    
    -- 2. Add Time-Series Features (Lag Logic)
    weather_features AS (
        SELECT 
            *,
            -- Convert integer flag to boolean for readability
            CAST(is_precipitating_int AS BOOLEAN) as is_precipitating,
            
            -- Feature: "Rain Started This Hour"
            -- (True if raining NOW, but was NOT raining PREVIOUS hour)
            CASE 
                WHEN is_precipitating_int = 1 
                     AND LAG(is_precipitating_int, 1, 0) OVER (PARTITION BY station_code ORDER BY weather_hour) = 0 
                THEN TRUE
                ELSE FALSE
            END as rain_started_this_hour
            
        FROM weather_hourly
    )
    
    -- 3. Join with the Disruption-Enriched Data
    SELECT 
        base.*,
        
        -- Weather Columns (Safe defaults for missing data)
        COALESCE(w.temp_c, 0) as temp_c,
        COALESCE(w.soil_temp_c, 0) as soil_temp_c,
        COALESCE(w.wind_speed_kmh, 0) as wind_speed_kmh,
        COALESCE(w.wind_gust_kmh, 0) as wind_gust_kmh,
        
        COALESCE(w.rain_mm, 0) as rain_mm,
        COALESCE(w.snow_cm, 0) as snow_cm,
        COALESCE(w.snow_depth_m, 0) as snow_depth_m,
        
        COALESCE(w.is_precipitating, FALSE) as is_precipitating,
        COALESCE(w.rain_started_this_hour, FALSE) as rain_started_this_hour
        
    FROM hourly_stats_enriched base
    LEFT JOIN weather_features w
        ON base.hour_bin = w.weather_hour 
        AND base.station_code = w.station_code
    ORDER BY base.hour_bin, base.station_code
""")

# --- Verification ---
print("✅ Table 'hourly_stats_final' created with all weather variables.")

# Check: Show me an hour with snow or high wind
print("\n--- Sample: Extreme Weather Hours ---")
check_query = """
    SELECT hour_bin, station_code, temp_c, snow_cm, wind_gust_kmh, soil_temp_c
    FROM hourly_stats_final
    WHERE snow_cm > 0 OR wind_gust_kmh > 40
    LIMIT 5
"""
display(con.execute(check_query).df())

Step 5: Integrating detailed weather data and calculating lag features...
✅ Table 'hourly_stats_final' created with all weather variables.

--- Sample: Extreme Weather Hours ---


,hour_bin,station_code,temp_c,snow_cm,wind_gust_kmh,soil_temp_c
0,2019-01-01 02:00:00,AKM,7.6175,0.0,43.919998,7.9175
1,2019-01-01 02:00:00,ALMB,7.8805,0.0,41.039997,7.8805
2,2019-01-01 02:00:00,ALMO,7.8935,0.0,41.039997,7.8935
3,2019-01-01 02:00:00,ALMP,7.9000,0.0,41.039997,7.9000
4,2019-01-01 02:00:00,AMR,8.2370,0.0,47.880001,8.1370


In [ ]:
# Check: Show me an hour where rain started
print("\n--- Sample: Rain Onset Events ---")
check_query = """
    SELECT *
    FROM hourly_stats_final
    LIMIT 5
"""
display(con.execute(check_query).df())


--- Sample: Rain Onset Events ---


,hour_bin,station_code,hour_of_day,day_of_week,month_of_year,total_trains,delayed_trains_count,total_delay_minutes,avg_delay_minutes,disruption_count,is_disrupted,temp_c,soil_temp_c,wind_speed_kmh,wind_gust_kmh,rain_mm,snow_cm,snow_depth_m,is_precipitating,rain_started_this_hour
0,2019-01-01 02:00:00,AC,2,2,1,0,0.0,0.0,0.0,0,False,8.087001,7.8370,21.542923,36.719997,0.0,0.0,0.0,False,False
1,2019-01-01 02:00:00,AH,2,2,1,0,0.0,0.0,0.0,0,False,7.361000,7.7110,19.813087,36.360001,0.0,0.0,0.0,False,False
2,2019-01-01 02:00:00,AHP,2,2,1,0,0.0,0.0,0.0,0,False,7.439000,7.7890,19.813087,36.360001,0.0,0.0,0.0,False,False
3,2019-01-01 02:00:00,AHPR,2,2,1,0,0.0,0.0,0.0,0,False,7.491000,7.8410,19.813087,36.360001,0.0,0.0,0.0,False,False
4,2019-01-01 02:00:00,AHZ,2,2,1,0,0.0,0.0,0.0,0,False,7.484500,7.8345,19.191748,35.639999,0.0,0.0,0.0,False,False


### Adding Holidays data (could be moved earlier for speed)

In [ ]:
# --- Cell: Step 6 - Add Calendar, NS Rush Hour & Holiday Features ---

print("Step 6: Integrating Holidays table and applying NS-specific Rush Hour/Weekend logic...")

con.execute("""
    CREATE OR REPLACE TABLE training_data_complete AS
    
    SELECT 
        t.*,
        
        -- 1. Holiday Integration
        -- Join on the DATE part of the timestamp
        CASE 
            WHEN h.holiday_name IS NOT NULL THEN TRUE 
            ELSE FALSE 
        END as is_holiday,
        
        -- 3. NS Specific Weekend Definition
        -- Definition: Friday 18:30 to Monday 04:00
        CASE 
            -- Friday (5) from 18:00 onwards (covers the 18:30 start)
            WHEN EXTRACT('dow' FROM t.hour_bin) = 5 AND EXTRACT('hour' FROM t.hour_bin) >= 18 THEN TRUE
            -- Saturday (6) and Sunday (0) are always weekend
            WHEN EXTRACT('dow' FROM t.hour_bin) IN (0, 6) THEN TRUE
            -- Monday (1) before 04:00
            WHEN EXTRACT('dow' FROM t.hour_bin) = 1 AND EXTRACT('hour' FROM t.hour_bin) < 4 THEN TRUE
            ELSE FALSE
        END as is_weekend,
        
        -- 4. NS Specific Rush Hour Definition
        -- Definition: Weekdays 06:30-09:00 and 16:00-18:30
        -- Note: Excludes Weekends (Sat/Sun) and Holidays (usually treated as Sunday schedule)
        CASE 
            -- If it's the weekend or a holiday, it is NOT rush hour
            WHEN (EXTRACT('dow' FROM t.hour_bin) IN (0, 6)) OR (h.holiday_name IS NOT NULL) THEN FALSE
            
            -- Morning Peak: 06:30 - 09:00
            -- Includes hour 6 (06:30 start), 7, 8. Excludes 9 (ends 09:00).
            WHEN EXTRACT('hour' FROM t.hour_bin) IN (6, 7, 8) THEN TRUE
            
            -- Evening Peak: 16:00 - 18:30
            -- Includes hour 16, 17, 18 (ends 18:30).
            WHEN EXTRACT('hour' FROM t.hour_bin) IN (16, 17, 18) THEN TRUE
            
            ELSE FALSE 
        END as is_rush_hour

    FROM hourly_stats_final t
    -- Left Join to Holidays Table on the specific date
    LEFT JOIN raw_holidays h
        ON CAST(t.hour_bin AS DATE) = CAST(h.date AS DATE)
    
    ORDER BY t.hour_bin, t.station_code
""")

# --- Verification ---
print("✅ Table 'training_data_complete' created.")

# Check 1: Verify Holiday Match
print("\n--- Sample: Holiday Matches ---")
check_holidays = """
    SELECT CAST(hour_bin AS DATE) as date, is_holiday, is_rush_hour
    FROM training_data_complete
    WHERE is_holiday = TRUE
    LIMIT 3
"""
display(con.execute(check_holidays).df())


Step 6: Integrating Holidays table and applying NS-specific Rush Hour/Weekend logic...
✅ Table 'training_data_complete' created.

--- Sample: Holiday Matches ---


,date,is_holiday,is_rush_hour
0,2019-01-01,True,False
1,2019-01-01,True,False
2,2019-01-01,True,False



--- Sample: Friday Evening Transition (Weekend Start) ---


,hour_bin,day_of_week,hour_of_day,is_weekend
0,2019-01-04 16:00:00,5,16,False
1,2019-01-04 16:00:00,5,16,False
2,2019-01-04 16:00:00,5,16,False
3,2019-01-04 16:00:00,5,16,False
4,2019-01-04 16:00:00,5,16,False


In [ ]:
# Check 2: Verify NS Weekend Logic (Friday Evening)
print("\n--- Sample: Friday Evening Transition (Weekend Start) ---")
check_weekend = """
    SELECT hour_bin, day_of_week, hour_of_day, is_weekend, is_rush_hour
    FROM training_data_complete
    WHERE day_of_week = 5 AND hour_of_day BETWEEN 16 AND 20
    LIMIT 5
"""
display(con.execute(check_weekend).df())


--- Sample: Friday Evening Transition (Weekend Start) ---


,hour_bin,day_of_week,hour_of_day,is_weekend,is_rush_hour
0,2019-01-04 16:00:00,5,16,False,True
1,2019-01-04 16:00:00,5,16,False,True
2,2019-01-04 16:00:00,5,16,False,True
3,2019-01-04 16:00:00,5,16,False,True
4,2019-01-04 16:00:00,5,16,False,True


### Adding station connections

In [ ]:
#todo gnn only probs

## Save full dataset

In [ ]:
# --- Cell: Split Data by Year (Rolling Window) ---

print("Assigning Train/Validation/Test splits based on last 2 years...")

# 1. Add the split column (if not exists)
try:
    con.execute("ALTER TABLE training_data_complete ADD COLUMN dataset_group VARCHAR")
except:
    pass # Column might already exist

# 2. Update based on the MAX date in your dataset
# Logic: 
#   - Test:       [Max Date - 1 Year]  to  [Max Date]
#   - Validation: [Max Date - 2 Years] to  [Max Date - 1 Year]
#   - Train:      Everything older
split_query = """
    UPDATE training_data_complete
    SET dataset_group = CASE
        -- The Test Set (The most recent 1 year)
        WHEN hour_bin >= (SELECT MAX(hour_bin) - INTERVAL 1 YEAR FROM training_data_complete) 
            THEN 'test'
        
        -- The Validation Set (The year before that)
        WHEN hour_bin >= (SELECT MAX(hour_bin) - INTERVAL 2 YEAR FROM training_data_complete) 
            THEN 'validation'
            
        -- The Training Set (Everything else)
        ELSE 'train'
    END
"""

con.execute(split_query)

# --- Verification ---
# It is CRITICAL to verify the start/end dates for each group
stats_query = """
    SELECT 
        dataset_group,
        COUNT(*) as rows,
        MIN(hour_bin) as start_date,
        MAX(hour_bin) as end_date,
        
        -- Calculate rough duration in days to verify
        DATE_DIFF('day', MIN(hour_bin), MAX(hour_bin)) as duration_days
    FROM training_data_complete
    GROUP BY dataset_group
    ORDER BY MIN(hour_bin)
"""
display(con.execute(stats_query).df())

print("✅ Splits assigned. Ready to save.")

Assigning Train/Validation/Test splits based on last 2 years...


,dataset_group,rows,start_date,end_date,duration_days
0,train,13920408,2019-01-01 02:00:00,2023-01-01 01:00:00,1461
1,validation,3477720,2023-01-01 02:00:00,2024-01-01 01:00:00,365
2,test,3487645,2024-01-01 02:00:00,2025-01-01 02:00:00,366


✅ Splits assigned. Ready to save.


In [ ]:
import os

print("Saving final training data to Parquet...")

# 1. Define the directory
# (Assuming filtered_output_root_folder is defined earlier, e.g., "data/")
full_dataset_dir = os.path.join(filtered_output_root_folder, "full_dataset")

# 2. Create the folder if it doesn't exist
os.makedirs(full_dataset_dir, exist_ok=True)

# 3. Define the full target file path
# This handles the slashes for you automatically
target_file = os.path.join(full_dataset_dir, "training_data_final.parquet")

print(f"Target path: {target_file}")

# 4. Save
# We inject the complete 'target_file' path into the query
save_query = f"""
    COPY training_data_complete 
    TO '{target_file}' 
    (FORMAT PARQUET, COMPRESSION SNAPPY, OVERWRITE_OR_IGNORE true)
"""

con.execute(save_query)
print("✅ File saved successfully.")

Saving final training data to Parquet...
Target path: c:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao\downloads\data\filtered\full_dataset\training_data_final.parquet
✅ File saved successfully.


In [ ]:
import os

print("Saving training data partitioned by split...")

full_dataset_dir = os.path.join(filtered_output_root_folder, "full_dataset_partitioned")
os.makedirs(full_dataset_dir, exist_ok=True)

# Note the 'PARTITION_BY' clause
con.execute(f"""
    COPY training_data_complete 
    TO '{full_dataset_dir}' 
    (FORMAT PARQUET, COMPRESSION SNAPPY, PARTITION_BY dataset_group, OVERWRITE_OR_IGNORE true)
""")

print(f"✅ Successfully saved partitioned data to: {full_dataset_dir}")

Saving training data partitioned by split...
✅ Successfully saved partitioned data to: c:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao\downloads\data\filtered\full_dataset_partitioned


In [ ]:
# This returns a DataFrame with column_name, column_type, null, key, default, and extra
con.execute("DESCRIBE training_data_complete").df()

,column_name,column_type,null,key,default,extra
0,hour_bin,TIMESTAMP,YES,None,None,None
1,station_code,VARCHAR,YES,None,None,None
2,hour_of_day,BIGINT,YES,None,None,None
3,day_of_week,BIGINT,YES,None,None,None
4,month_of_year,BIGINT,YES,None,None,None
5,total_trains,BIGINT,YES,None,None,None
6,delayed_trains_count,HUGEINT,YES,None,None,None
7,total_delay_minutes,HUGEINT,YES,None,None,None
8,avg_delay_minutes,DOUBLE,YES,None,None,None
9,disruption_count,BIGINT,YES,None,None,None


In [ ]:
con.execute("""
    SELECT 
        station_code, 
        hour_bin, 
        COUNT(*) as count
    FROM training_data_complete
    GROUP BY station_code, hour_bin
    HAVING COUNT(*) > 1
    ORDER BY count DESC
    LIMIT 20
""").df()

,station_code,hour_bin,count
